In [29]:
import os
# import tempfile

import numpy as np

# os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "mpl-cache"))

import matplotlib
import matplotlib.pyplot as plt
# if not os.environ.get("DISPLAY"):
#     matplotlib.use("Agg")


In [30]:
def sigmoid(t):
    t = np.clip(t, -500.0, 500.0)
    return 1.0 / (1.0 + np.exp(-t))


def generate_data(n_samples, w_true, b_true, mu_true, seed=42):
    rng = np.random.default_rng(seed)

    x = rng.normal(loc=0.0, scale=1.5, size=n_samples)
    logits = w_true * x + b_true
    probs = sigmoid(logits)
    y = rng.binomial(n=1, p=probs)

    z = rng.normal(loc=0.0, scale=1.0, size=n_samples)
    positive_mask = y == 1
    z[positive_mask] = rng.normal(
        loc=mu_true, scale=1.0, size=positive_mask.sum()
    )

    return x, y, z


In [31]:
def estimate_mu(y, z):
    positive_count = y.sum()
    if positive_count == 0:
        raise ValueError("Cannot estimate mu because there are no positive samples.")
    return np.sum(y * z) / positive_count


def logistic_log_likelihood(x, y, w, b):
    logits = w * x + b
    return np.sum(y * logits - np.logaddexp(0.0, logits))


In [32]:
def train_logistic_regression(x, y, learning_rate=0.01, epochs=2000):
    w = 0.0
    b = 0.0
    log_likelihood_history = []

    n_samples = x.shape[0]

    for epoch in range(epochs):
        sigma = sigmoid(w * x + b)
        grad_w = np.sum(x * (y - sigma))
        grad_b = np.sum(y - sigma)

        w += learning_rate * grad_w / n_samples
        b += learning_rate * grad_b / n_samples

        ll = logistic_log_likelihood(x, y, w, b)
        log_likelihood_history.append(ll)

        if epoch % 200 == 0 or epoch == epochs - 1:
            print(
                f"Epoch {epoch:4d} | log-likelihood = {ll:.4f} | "
                f"w = {w:.4f}, b = {b:.4f}"
            )

    return w, b, np.array(log_likelihood_history)


In [37]:
def plot_training_curve(log_likelihood_history):
    if plt is None or matplotlib is None:
        print("matplotlib is not installed, so the training curve was not plotted.")
        return

    plt.figure(figsize=(8, 5))
    plt.plot(log_likelihood_history, color="tab:blue", linewidth=2)
    plt.title("Log-Likelihood During Training")
    plt.xlabel("Epoch")
    plt.ylabel("Log-Likelihood")
    plt.grid(alpha=0.3)
    plt.tight_layout()

    if matplotlib.get_backend().lower() == "agg":
        output_path = "joint_model_log_likelihood.png"
        plt.show()
        plt.savefig(output_path, dpi=150)
        print(f"Saved training curve to {output_path}")
    else:
        plt.show()


In [34]:
n_samples = 1000
w_true = 2.0
b_true = -0.7
mu_true = 2.5

learning_rate = 0.1
epochs = 3000

x, y, z = generate_data(
    n_samples=n_samples,
    w_true=w_true,
    b_true=b_true,
    mu_true=mu_true,
    seed=42,
)


In [35]:
mu_est = estimate_mu(y, z)
w_est, b_est, log_likelihood_history = train_logistic_regression(
    x=x,
    y=y,
    learning_rate=learning_rate,
    epochs=epochs,
)

print("\nParameter comparison")
print(f"w_true  = {w_true: .4f} | w_est  = {w_est: .4f}")
print(f"b_true  = {b_true: .4f} | b_est  = {b_est: .4f}")
print(f"mu_true = {mu_true: .4f} | mu_est = {mu_est: .4f}")


Epoch    0 | log-likelihood = -669.1274 | w = 0.0489, b = -0.0088
Epoch  200 | log-likelihood = -373.6586 | w = 1.6668, b = -0.5403
Epoch  400 | log-likelihood = -370.9665 | w = 1.8610, b = -0.6369
Epoch  600 | log-likelihood = -370.7552 | w = 1.9164, b = -0.6640
Epoch  800 | log-likelihood = -370.7344 | w = 1.9339, b = -0.6724
Epoch 1000 | log-likelihood = -370.7322 | w = 1.9396, b = -0.6752
Epoch 1200 | log-likelihood = -370.7319 | w = 1.9414, b = -0.6761
Epoch 1400 | log-likelihood = -370.7319 | w = 1.9420, b = -0.6763
Epoch 1600 | log-likelihood = -370.7319 | w = 1.9422, b = -0.6764
Epoch 1800 | log-likelihood = -370.7319 | w = 1.9423, b = -0.6765
Epoch 2000 | log-likelihood = -370.7319 | w = 1.9423, b = -0.6765
Epoch 2200 | log-likelihood = -370.7319 | w = 1.9423, b = -0.6765
Epoch 2400 | log-likelihood = -370.7319 | w = 1.9423, b = -0.6765
Epoch 2600 | log-likelihood = -370.7319 | w = 1.9423, b = -0.6765
Epoch 2800 | log-likelihood = -370.7319 | w = 1.9423, b = -0.6765
Epoch 2999

In [38]:
plot_training_curve(log_likelihood_history)


Saved training curve to joint_model_log_likelihood.png


/var/folders/27/h86gj_fd3xg23kmxw_95c6kc0000gn/T/ipykernel_8379/2549995449.py:16: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
